In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}}
)

## 1、设置agent名称
创建agent时，LangChain允许指定名称。

### 1.1 用法

In [ ]:
agent = create_agent(
    model=model,
    name="agent-test",
)

response = agent.invoke({"messages": ["hello"]})
for message in response["messages"]:
    message.pretty_print()

### 1.2 经典使用场景
name 在 Multi-Agent 场景中最常被提及，用于区分不同的 Agent。但它的作用并不局限于多 Agent编排。在实际工程中，出现如下场景，通常都建议为 Agent 设置一个清晰且稳定的 name 。

1. 流式输出归因

在启用流式输出时，name 可用于 标识当前输出内容来自哪个 Agent 。

这在多 Agent 协作、Agent 嵌套调用，或前端需要实时展示不同执行主体输出时尤其有用，便于准确区分 token 或事件的来源。

2. 消息身份标记

设置 name 后，Agent 产生的 AIMessage 会携带对应的name信息 。

这使得系统在 保存会话记录 、 回放执行过程 、 构建审计日志 或 前端展示消息角色 时，能够明确识别消息的生成者。

3. 调试与trace可读性

在调试、日志分析和链路追踪过程中，name 可以作为 Agent 的稳定标识，帮助开发者 快速判断当前执行的是哪个 Agent 。

当系统中存在多个能力相近的 Agent，或一个 Agent 被嵌套在更复杂的工作流中时，名称能够显著提升trace 的可读性 和 问题定位 效率。

4. 组件化封装

在工程实践中，Agent 常被封装为可复用的能力模块，例如检索助手、SQL 助手、报告生成助手等。

为 Agent 设置 name ，有助于在模块注册、运行监控、日志归档和能力复用时保持一致的身份标识。

如果后续需要将该 Agent 进一步作为子图节点、工具能力或子模块接入更复杂系统，也能降低维护和迁移成本。

5. 前端展示与运行态可观测性

在带有可视化界面的应用中，name 还可以直接作为运行时 展示标识 使用。

例如，在执行面板中显示“当前活跃 Agent”“本轮输出来源”或“调用链路中的执行节点”时，name 能帮助开发者和用户更直观地理解系统当前的执行状态。

6. 作为稳定的运行时身份标识

从更通用的角度看，name 可以理解为 Agent 在系统中的“ 运行时身份 ID ”。

相比临时性的展示名称，一个稳定、规范的 name 更适合用于 日志检索 、 监控统计 、 链路分析 和跨模块协作 ，因此在生产环境中通常建议显式设置，而不是依赖默认行为。

## 2、系统提示词

使用create_agent创建Agent时，需传入模型和工具，可选的传入系统提示词。**提示词为Agent提供了人任务背景、行为准则和操作指南。**

系统指令，即SystemMessage，通过system_prompt设置，定义Agent行为。这个参数可以是str或者SystemMessage类型。

使用建议：
- 明确说明Agent的角色。
- 定义输出格式。
- 说明何时使用工具。

比如：

In [ ]:
from langchain_core.tools import tool

flag = 0

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    返回对应城市的天气。

    Args:
        city: 城市
    """
    global flag
    flag += 1

    if flag < 3:
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"

    return f"{city}天气晴朗"

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="""
    你是天气助手。

    工作流程：
    1、理解用户的城市查询。
    2、使用get_weather工具获取数据。
    3、简洁清晰的回答。

    输出格式：
    - 天气情况。
    - 温度。
    - 注意事项（如有）
    """
)

提示词有两种方式：静态设置和动态设置。动态设置需要借助中间件。

举例：

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    HumanMessage("今天广州的天气如何")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)

response = agent.invoke({"messages": messages})

for message in response["messages"]:
    message.pretty_print()